# Aff-Wild2 Stage 3 — defence-time stretch (items 1–3 of the v3 plan §7 follow-up list)

Three independent items; rerun only the ones whose inputs changed.

1. **F4 with `window=10` × three seeds.** The Section 4 sweep in `aw2_06` flagged `w=10` as the strongest single-row F4 configuration (raw `1.328`, smoothed `1.336`), but only at seed 42. With the multi-seed pool we measured at `w=5`, the `w=10` headline number is still single-seed. Re-train at seeds 0/1/2 and report mean ± s.d.
2. **F6+ candidate variants** (queued in the v3 plan §4 roadmap): F6a Dirichlet-weighted ensemble (post-hoc, no training), F6c IACA-gate (inconsistency-aware gate on top of F4), F6d MBT (attention-bottleneck fusion).
3. **Stage 1 EXPR-head fix on `enet_b0_8_va_mtl`.** Sprint A landed the enet Stage-1 reproduction $-0.031~P_{\mathrm{MTL}}$ short of Paper A, driven entirely by `F1_EXPR_macro` ($0.337$ vs.\ Paper-A $0.504$). Cross-entropy and macro-F1 disagree under class imbalance; the patch in `src/train.py` adds `head.expr_select_by: f1_macro` which selects `best.pt` on validation macro-F1 instead of CE loss.

Prerequisites:
* All artifacts produced by `aw2_05_stage3_fusion.ipynb` (F1–F5 + visual_only + audio_only checkpoints).
* `aw2_06_stage3_ablations.ipynb` Section 1 has produced the smoothing artifacts for the F4 baseline.

In [1]:
import json, os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

ABL = REPO / 'results' / 'aw2_stage3_ablations'
ABL.mkdir(parents=True, exist_ok=True)
RUNTIME_CFG = ABL / '_configs'
RUNTIME_CFG.mkdir(exist_ok=True)
print('cwd          :', Path.cwd())
print('extras out   :', ABL)

cwd          : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code
extras out   : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\aw2_stage3_ablations


In [2]:
def _run(cmd):
    print('$', ' '.join(map(str, cmd)))
    proc = subprocess.run(cmd, check=False)
    assert proc.returncode == 0, f'command failed (exit={proc.returncode}): {cmd}'

def run_train(config_rel):
    _run([sys.executable, '-u', 'train_fusion.py', '--config', str(config_rel)])

def run_train_stage1(config_rel):
    _run([sys.executable, '-u', 'train.py', '--config', str(config_rel)])

def run_eval(checkpoint, config_rel):
    _run([sys.executable, '-u', 'eval_fusion.py',
          '--config', str(config_rel),
          '--checkpoint', str(checkpoint)])

def run_smooth(checkpoint, config_rel, output_dir=None):
    cmd = [sys.executable, '-m', 'src.smooth_fusion',
           '--checkpoint', str(checkpoint),
           '--config', str(config_rel)]
    if output_dir is not None:
        cmd += ['--output-dir', str(output_dir)]
    _run(cmd)

def train_eval_smooth(config_rel):
    from omegaconf import OmegaConf
    cfg = OmegaConf.load(config_rel)
    out = REPO / cfg.output.results_dir
    run_train(config_rel)
    run_eval(out / 'best.pt', config_rel)
    run_smooth(out / 'best.pt', config_rel)
    return out

## 1. F4 with `window=10` × three seeds

Base config: `configs/stage3_f4_xattn_w10_wd1e3.yaml` (window=10, wd=1e-3, 3 epochs). Each seed gets its own runtime YAML under `results/aw2_stage3_ablations/_configs/`. Same protocol as `aw2_06` Section 5 (the wd1e3-protocol three-seed sweep at window=5).

In [3]:
from omegaconf import OmegaConf

BASE_CFG = REPO / 'configs' / 'stage3_f4_xattn_w10_wd1e3.yaml'
for seed in (0, 1, 2):
    cfg = OmegaConf.load(BASE_CFG)
    cfg.seed = seed
    cfg.run_name = f'stage3_f4_xattn_w10_seed{seed}'
    cfg.output.results_dir = f'results/stage3_f4_xattn_w10_seed{seed}'
    out_cfg = RUNTIME_CFG / f'stage3_f4_xattn_w10_seed{seed}.yaml'
    OmegaConf.save(cfg, out_cfg)
    train_eval_smooth(out_cfg.relative_to(REPO).as_posix())

# Also run the seed=42 baseline once if not already done.
if not (REPO / 'results' / 'stage3_f4_xattn_w10_wd1e3' / 'best.pt').exists():
    train_eval_smooth('configs/stage3_f4_xattn_w10_wd1e3.yaml')

$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config results/aw2_stage3_ablations/_configs/stage3_f4_xattn_w10_seed0.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u eval_fusion.py --config results/aw2_stage3_ablations/_configs/stage3_f4_xattn_w10_seed0.yaml --checkpoint C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\stage3_f4_xattn_w10_seed0\best.pt
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -m src.smooth_fusion --checkpoint C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\stage3_f4_xattn_w10_seed0\best.pt --config results/aw2_stage3_ablations/_configs/stage3_f4_xattn_w10_seed0.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config results/aw2_stage3_ablations/_configs/stage3_f4_xattn_w10_seed1.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u eval_fusion.py --config results/aw2_stage3_ablations

## 2. Stage 1 EXPR-head fix on `enet_b0_8_va_mtl`

The Sprint A enet checkpoint was selected by minimum class-weighted CE on val. CE and macro-F1 are weakly correlated under the AffWild2 EXPR class imbalance (the 'Other' bucket dominates), so CE-best does not coincide with F1-best. The patch in `src/train.py` adds an optional `score_fn` for `_train_head`; setting `head.expr_select_by: f1_macro` in the config wires a macro-F1 scorer in for the EXPR head. The new config `configs/aw2_stage1_enet_f1.yaml` is otherwise identical to `aw2_stage1_enet.yaml`.

Expected behaviour:
* New `best.pt` lands on the epoch with the highest val macro-F1 (often a different epoch than the CE-best).
* Headline F1_EXPR up by some amount; CCC_VA and F1_AU unchanged (they are still selected by their own val-loss criteria).
* Smoothing-time `P_MTL_best` should also rise.

In [4]:
run_train_stage1('configs/aw2_stage1_enet_f1.yaml')

$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train.py --config configs/aw2_stage1_enet_f1.yaml


In [5]:
# Re-run the smoothing grid + eval helper from aw2_02 against the new checkpoint.
# Stage 1 has its own evaluator; reuse the smoothing_grid pattern from
# aw2_02_stage1_train_eval.ipynb. Self-contained inline implementation:
import numpy as np
import torch
from omegaconf import OmegaConf

from src.heads.mtl_head import MTLHead, MTLHeadConfig
from src.smoothing import gaussian_smooth_per_video
from src.train import load_split
from src.utils.metrics import (
    f1_macro_au, metric_for_Exp, metric_for_VA, p_mtl,
)

def _stage1_predict(checkpoint, config_path):
    cfg = OmegaConf.load(config_path)
    device = cfg.train.device if torch.cuda.is_available() else 'cpu'
    X_val, anno = load_split(cfg.data.val_annotations, cfg.features_cache)
    ck = torch.load(checkpoint, map_location='cpu', weights_only=False)
    head_cfg = MTLHeadConfig(**ck['head_cfg'])
    model = MTLHead(head_cfg).to(device).eval()
    model.load_state_dict(ck['state_dict'])
    pred_expr, pred_va, pred_au = [], [], []
    with torch.no_grad():
        xt = torch.from_numpy(X_val.astype(np.float32))
        for i in range(0, xt.size(0), 4096):
            e, v, a = model(xt[i:i+4096].to(device))
            pred_expr.append(torch.softmax(e, dim=-1).cpu().numpy())
            pred_va.append(v.cpu().numpy())
            pred_au.append(a.cpu().numpy())
    return anno, np.concatenate(pred_expr), np.concatenate(pred_va), np.concatenate(pred_au), head_cfg

def _stage1_metrics(anno, pe, pv, pa, head_cfg):
    em = anno.mask_expr == 1
    f1_expr, _, _ = metric_for_Exp(anno.y_expr[em], pe.argmax(axis=1)[em],
                                   class_num=head_cfg.num_expr)
    vm = anno.mask_va == 1
    cV, cA, cVA = metric_for_VA(anno.y_va[vm, 0], anno.y_va[vm, 1], pv[vm, 0], pv[vm, 1])
    am = anno.mask_au == 1
    f1_au = f1_macro_au(anno.y_aus[am], pa[am], threshold=0.5)
    return cV, cA, cVA, f1_expr, f1_au, p_mtl(cVA, f1_expr, f1_au)

def stage1_smoothing_grid(checkpoint, config_path,
                          sigmas=(0.1, 1, 10, 50, 100, 500, 1e3, 1e4, 1e5),
                          deltas=(1, 5, 10, 50, 100)):
    anno, pe, pv, pa, head_cfg = _stage1_predict(checkpoint, config_path)
    raw = _stage1_metrics(anno, pe, pv, pa, head_cfg)
    best_p, best_key = raw[-1], ('raw', None, None)
    for sigma in sigmas:
        for delta in deltas:
            sm_e = gaussian_smooth_per_video(pe, anno.videoname_frames, sigma=sigma, delta=delta)
            sm_v = gaussian_smooth_per_video(pv, anno.videoname_frames, sigma=sigma, delta=delta)
            m = _stage1_metrics(anno, sm_e, sm_v, pa, head_cfg)
            if m[-1] > best_p:
                best_p = m[-1]; best_key = ('sm', sigma, delta)
    return raw, best_key, best_p

OLD_CKPT = REPO / 'results' / 'aw2_stage1_enet'    / 'best.pt'
NEW_CKPT = REPO / 'results' / 'aw2_stage1_enet_f1' / 'best.pt'

for label, ckpt, cfg in [('old (val_loss)', OLD_CKPT, 'configs/aw2_stage1_enet.yaml'),
                          ('new (f1_macro)', NEW_CKPT, 'configs/aw2_stage1_enet_f1.yaml')]:
    if not Path(ckpt).exists():
        print(f'[skip] {label}: missing {ckpt}')
        continue
    raw, best_key, best_p = stage1_smoothing_grid(ckpt, cfg)
    cV, cA, cVA, f1_e, f1_a, p = raw
    print(f'{label:18s}  CCC_V={cV:.4f}  CCC_A={cA:.4f}  F1_EXPR={f1_e:.4f}  F1_AU={f1_a:.4f}  P_raw={p:.4f}  P_smooth={best_p:.4f}  best={best_key}')

old (val_loss)      CCC_V=0.4640  CCC_A=0.4211  F1_EXPR=0.3369  F1_AU=0.4796  P_raw=1.2591  P_smooth=1.3688  best=('sm', 500, 10)
new (f1_macro)      CCC_V=0.4640  CCC_A=0.4211  F1_EXPR=0.3369  F1_AU=0.4796  P_raw=1.2591  P_smooth=1.3688  best=('sm', 500, 10)


## 3. F6c IACA-gate (inconsistency-aware gate on top of F4)

Adds a 2-layer MLP gate that reads `[pooled_v, pooled_a, |pooled_v - pooled_a|]` and emits a per-task gate `g ∈ [0,1]^4`. Output per task `t` is `logit_t = g_t · fused_t + (1 - g_t) · visual_t`. Falls back to visual-only when modalities disagree. Praveen et al. CVPRW 2024.

In [6]:
train_eval_smooth('configs/stage3_f6c_iaca.yaml')

$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f6c_iaca.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u eval_fusion.py --config configs/stage3_f6c_iaca.yaml --checkpoint C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\stage3_f6c_iaca\best.pt
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -m src.smooth_fusion --checkpoint C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\stage3_f6c_iaca\best.pt --config configs/stage3_f6c_iaca.yaml


WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/results/stage3_f6c_iaca')

## 4. F6d MBT (attention-bottleneck fusion)

B=4 latent bottleneck tokens mediate inter-modal information exchange (Nagrani et al. NeurIPS 2021). At `hidden=192` the variant fits within the 2 M param ceiling (the four `_CrossAttnBlock` instances dominate the cost).

In [7]:
train_eval_smooth('configs/stage3_f6d_mbt.yaml')

$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u train_fusion.py --config configs/stage3_f6d_mbt.yaml
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -u eval_fusion.py --config configs/stage3_f6d_mbt.yaml --checkpoint C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\stage3_f6d_mbt\best.pt
$ c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Scripts\python.exe -m src.smooth_fusion --checkpoint C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\stage3_f6d_mbt\best.pt --config configs/stage3_f6d_mbt.yaml


WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/results/stage3_f6d_mbt')

## 5. F6a Dirichlet-weighted ensemble (post-hoc)

No training. Loads the F1, F2, F3, F4, F5 + `visual_only` checkpoints, runs one validation forward pass each, and grid-searches a per-task simplex weight vector `w ∈ Δ^N` (step 0.2 → 126 points over 5 components). Reference: SUN team CVPRW 2024.

Note: F4 takes a temporal window dataset; the implementation in `src.eval_fusion.evaluate_f6a_dirichlet` builds a per-variant val loader using `wants_window`, so all variants align by row even when one wants windows and another wants per-frame. We exclude F4 from this F6a sweep specifically because its row order would diverge if the underlying dataset construction were reused; F6a is therefore the late-blend over the per-frame-only variants {F1, F2, F3, F5, visual_only}.

In [8]:
from src.eval_fusion import evaluate_f6a_dirichlet

checkpoints = {
    'visual_only': 'results/stage3_visual_only/best.pt',
    'f1_concat':   'results/stage3_f1_concat/best.pt',
    'f2_blend':    'results/stage3_f2_blend/best.pt',
    'f3_gate':     'results/stage3_f3_gate/best.pt',
    'f5_lmf':      'results/stage3_f5_lmf/best.pt',
}
metrics = evaluate_f6a_dirichlet(
    checkpoints=checkpoints,
    config_path='configs/stage3_f6a_dirichlet.yaml',
    step=0.2,
)
print('F6a Dirichlet ensemble:')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:>18s}  {v:.4f}')
    else:
        print(f'  {k:>18s}  {v}')

[stage3] wrote results\stage3_f6a_dirichlet\metrics_f6a.md
               ccc_V = 0.4654
               ccc_A = 0.4147
              CCC_VA = 0.4401
       F1_EXPR_macro = 0.3399
          F1_AU_best = 0.4896
           t_AU_best = 0.6000
               P_MTL = 1.7095
             variant = f6a_dirichlet
              w_expr = [0.2, 0.2, 0.6000000000000001, 0.0, 0.0]
                w_va = [0.6000000000000001, 0.2, 0.0, 0.2, 0.0]
                w_au = [0.0, 0.0, 0.2, 0.4, 0.4]
                t_au = 0.6000
            variants = ['visual_only', 'f1_concat', 'f2_blend', 'f3_gate', 'f5_lmf']
F6a Dirichlet ensemble:
               ccc_V  0.4654
               ccc_A  0.4147
              CCC_VA  0.4401
       F1_EXPR_macro  0.3399
          F1_AU_best  0.4896
           t_AU_best  0.6000
               P_MTL  1.7095
             variant  f6a_dirichlet
              w_expr  [0.2, 0.2, 0.6000000000000001, 0.0, 0.0]
                w_va  [0.6000000000000001, 0.2, 0.0, 0.2, 0.0]
             

## 6. Aggregate the new rows into a stretch-summary table

Walks every result dir produced above (Sections 1, 3, 4) and emits `results/aw2_stage3_ablations/summary_extras.{md,json}`. F6a sits inside its own MD output (`results/stage3_f6a_dirichlet/metrics_f6a.md`); we include it as a one-row block here for completeness.

In [10]:
EXTRA_VARIANTS = [
    ('F4 w=10 seed=42 (wd1e3)',  'stage3_f4_xattn_w10_wd1e3'),
    ('F4 w=10 seed=0',           'stage3_f4_xattn_w10_seed0'),
    ('F4 w=10 seed=1',           'stage3_f4_xattn_w10_seed1'),
    ('F4 w=10 seed=2',           'stage3_f4_xattn_w10_seed2'),
    ('F6c IACA-gate',            'stage3_f6c_iaca'),
    ('F6d MBT',                  'stage3_f6d_mbt'),
]

def _read_smoothing(results_dir):
    p = REPO / 'results' / results_dir / 'smoothing.json'
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding='utf-8'))

lines = ['# Aff-Wild2 Stage 3 — defence-time stretch summary (items 1–3)\n']
lines.append('| variant | raw CCC_VA | raw F1_EXPR | raw F1_AU | raw P_MTL@0.5 | best (sigma, delta) | sm CCC_VA | sm F1_EXPR | sm P_MTL@0.5 | delta_smooth |')
lines.append('| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |')
for label, rdir in EXTRA_VARIANTS:
    s = _read_smoothing(rdir)
    if s is None:
        lines.append(f'| `{label}` | --- (not run) | | | | | | | | |')
        continue
    r, b = s['raw'], s['best']
    sigdel = f"({b['sigma']:g}, {b['delta']})"
    delta = b['P_MTL@0.5'] - r['P_MTL@0.5']
    lines.append(
        f"| `{label}` | {r['CCC_VA']:.4f} | {r['F1_EXPR_macro']:.4f} | {r['F1_AU@0.5']:.4f} | "
        f"{r['P_MTL@0.5']:.4f} | {sigdel} | {b['CCC_VA']:.4f} | {b['F1_EXPR_macro']:.4f} | "
        f"{b['P_MTL@0.5']:.4f} | {delta:+.4f} |"
    )

f6a_md = REPO / 'results' / 'stage3_f6a_dirichlet' / 'metrics_f6a.md'
if f6a_md.exists():
    lines.append('\n## F6a Dirichlet ensemble (post-hoc, no training)\n')
    lines.append(f6a_md.read_text(encoding='utf-8'))

summary_md = '\n'.join(lines) + '\n'
out = ABL / 'summary_extras.md'
out.write_text(summary_md, encoding='utf-8')
print(summary_md)

# Aff-Wild2 Stage 3 — defence-time stretch summary (items 1–3)

| variant | raw CCC_VA | raw F1_EXPR | raw F1_AU | raw P_MTL@0.5 | best (sigma, delta) | sm CCC_VA | sm F1_EXPR | sm P_MTL@0.5 | delta_smooth |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `F4 w=10 seed=42 (wd1e3)` | 0.5270 | 0.3304 | 0.4705 | 1.3279 | (100000, 5) | 0.5316 | 0.3340 | 1.3362 | +0.0083 |
| `F4 w=10 seed=0` | 0.5370 | 0.2879 | 0.4710 | 1.2960 | (1000, 10) | 0.5494 | 0.2840 | 1.3044 | +0.0085 |
| `F4 w=10 seed=1` | 0.4833 | 0.2751 | 0.4636 | 1.2220 | (500, 10) | 0.4889 | 0.2784 | 1.2310 | +0.0089 |
| `F4 w=10 seed=2` | 0.4673 | 0.3011 | 0.4650 | 1.2334 | (100000, 10) | 0.4749 | 0.2974 | 1.2374 | +0.0039 |
| `F6c IACA-gate` | 0.4144 | 0.2704 | 0.4796 | 1.1644 | (1000, 5) | 0.4261 | 0.2716 | 1.1773 | +0.0129 |
| `F6d MBT` | 0.4822 | 0.2957 | 0.4705 | 1.2484 | (100, 10) | 0.4997 | 0.2955 | 1.2657 | +0.0173 |

## F6a Dirichlet ensemble (post-hoc, no training)

# Stage 3 metrics - variant=f6a_dir

Drop the rows from `summary_extras.md` into the §6.x.6 'Defence-time stretch' subsection of `internship_report.tex` once each section finishes; the headline rows feed Tables `tab:stage3aw2_ablations` (replace single-seed F4 w=10 row with mean ± s.d.) and the new F6 candidates table.

Stage 1 EXPR-fix outputs are dumped per the printed cell in Section 2; if the new checkpoint clears the $\pm 0.015~P_{\mathrm{MTL}}$ reproduction gate against Paper A's enet target ($1.290$ aligned), update Table~`tab:stage1aw2_delivered` and retire the residual EXPR shortfall noted in Section~\ref{sec:stage1aw2}.